# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [1]:
import sys
import time
import warnings

import pandas as pd
from IPython.core.display import Markdown
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import (
    GridSearchCV,
    cross_val_score,
    train_test_split,
)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
notebook_start_time = time.time()

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.

In [3]:
df = pd.read_csv("../data/day-of-week-not-scaled.csv")
df_days = pd.read_csv("../data/dayofweek.csv")

In [4]:
df["dayofweek"] = df_days["dayofweek"].values
df.head(3)

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,dayofweek
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4


2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [5]:
X = df.drop(columns="dayofweek")
y = df["dayofweek"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.

In [7]:
%%time
grid_search = GridSearchCV(
    estimator=SVC(random_state=21, probability=True),
    param_grid={
        "C": [0.01, 0.1, 1, 1.5, 5, 10],
        "kernel": ("linear", "rbf", "sigmoid"),
        "gamma": ("scale", "auto"),
        "class_weight": ("balanced", None),
    },
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 72 candidates, totalling 360 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.


[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    2.5s
[Parallel(n_jobs=-1)]: Done 168 tasks      | elapsed:   22.4s
[Parallel(n_jobs=-1)]: Done 360 out of 360 | elapsed:  2.7min finished


CPU times: user 1.97 s, sys: 770 ms, total: 2.74 s
Wall time: 2min 41s


GridSearchCV(estimator=SVC(probability=True, random_state=21), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 1.5, 5, 10],
                         'class_weight': ('balanced', None),
                         'gamma': ('scale', 'auto'),
                         'kernel': ('linear', 'rbf', 'sigmoid')},
             verbose=1)

In [8]:
all_params = grid_search.best_estimator_.get_params()
best_params = grid_search.best_params_.copy()  # только параметры из param_grid
best_params.update(
    {k: v for k, v in all_params.items() if k in ["random_state", "probability"]}
)
print("Лучшие параметры:", best_params)
print("Лучший результат:", round(grid_search.best_score_, 5))

Лучшие параметры: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Лучший результат: 0.87611


2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [9]:
# pd.DataFrame(grid_search.cv_results_).info()

In [10]:
best_params_df = pd.DataFrame(grid_search.cv_results_.copy()).sort_values(
    by="rank_test_score"
)
show_cols = [
    "rank_test_score",
    "mean_test_score",
    "std_test_score",
    "param_kernel",
    "param_C",
    "param_gamma",
    "param_class_weight",
]
best_params_df = best_params_df[show_cols]
best_params_df.head(5)

,rank_test_score,mean_test_score,std_test_score,param_kernel,param_C,param_gamma,param_class_weight
70,1,0.876109,0.018419,rbf,10,auto,None
64,2,0.863500,0.010870,rbf,10,auto,balanced
58,3,0.816018,0.008116,rbf,5,auto,None
52,4,0.808608,0.021007,rbf,5,auto,balanced
63,5,0.721052,0.034438,linear,10,auto,balanced


Между 1 и 2 местом разница по mean_test_score составляет всего 0.01261. Обе модели используют тяжелое ядро rbf и высокий штраф C=10.0, то есть вторую модель нельзя назвать более простой. Более простые модели (linear или с меньшим param C - усиление регуляризации) показывают значительно более низкие точности.
На мой взгляд альтернативой явл-ся только вторая модель с class_weight='balanced' за счет более низкого разброса дисперсии std = 0.0108.

## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.

In [11]:
grid_search_dt = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=21),
    param_grid={
        "max_depth": range(1, 50),
        "criterion": ("entropy", "gini"),
        "class_weight": ("balanced", None),
        # 'max_features': ['sqrt', 'log2'],
    },
    scoring="accuracy",
    n_jobs=-1,
)

grid_search_dt.fit(X_train, y_train)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=21), n_jobs=-1,
             param_grid={'class_weight': ('balanced', None),
                         'criterion': ('entropy', 'gini'),
                         'max_depth': range(1, 50)},
             scoring='accuracy')

In [12]:
all_params_dt = grid_search_dt.best_estimator_.get_params()
best_params_dt = grid_search_dt.best_params_.copy()  # только параметры из param_grid
best_params_dt["random_state"] = all_params_dt["random_state"]
print("Лучшие параметры:", best_params_dt)
print("Лучший результат:", round(grid_search_dt.best_score_, 5))

Лучшие параметры: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Лучший результат: 0.87386


2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [13]:
df_dt_results = pd.DataFrame(grid_search_dt.cv_results_).sort_values(
    by=["rank_test_score", "param_max_depth"]
)

show_cols_dt = [
    "rank_test_score",
    "mean_test_score",
    "std_test_score",
    "param_criterion",
    "param_max_depth",
    "param_class_weight",
]

df_dt_table = df_dt_results[show_cols_dt]
df_dt_table.head(5)

,rank_test_score,mean_test_score,std_test_score,param_criterion,param_max_depth,param_class_weight
69,1,0.873865,0.025066,gini,21,balanced
73,2,0.873854,0.025018,gini,25,balanced
70,3,0.872378,0.025263,gini,22,balanced
71,4,0.872372,0.025179,gini,23,balanced
75,4,0.872372,0.025179,gini,27,balanced


более простая версия с глубиной 21 выиграла по точности у более сложных с бОльшими глубинами.

## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.

In [14]:
grid_search_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=21),
    param_grid={
        "n_estimators": (5, 10, 50, 100),
        "max_depth": range(1, 50),
        "criterion": ("entropy", "gini"),
        "class_weight": ("balanced", None),
        # 'max_features': ['sqrt', 'log2'],
    },
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
)

grid_search_rf.fit(X_train, y_train)

Fitting 5 folds for each of 784 candidates, totalling 3920 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 512 tasks      | elapsed:    3.4s
[Parallel(n_jobs=-1)]: Done 1512 tasks      | elapsed:    9.9s
[Parallel(n_jobs=-1)]: Done 3680 tasks      | elapsed:   22.5s
[Parallel(n_jobs=-1)]: Done 3889 out of 3920 | elapsed:   23.6s remaining:    0.2s
[Parallel(n_jobs=-1)]: Done 3920 out of 3920 | elapsed:   23.9s finished


GridSearchCV(estimator=RandomForestClassifier(random_state=21), n_jobs=-1,
             param_grid={'class_weight': ('balanced', None),
                         'criterion': ('entropy', 'gini'),
                         'max_depth': range(1, 50),
                         'n_estimators': (5, 10, 50, 100)},
             scoring='accuracy', verbose=1)

In [15]:
all_params_rf = grid_search_rf.best_estimator_.get_params()
best_params_rf = grid_search_rf.best_params_.copy()  # только параметры из param_grid
best_params_rf["random_state"] = all_params_rf["random_state"]
print("Лучшие параметры:", best_params_rf)
print("Лучший результат:", round(grid_search_rf.best_score_, 5))

Лучшие параметры: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 24, 'n_estimators': 100, 'random_state': 21}
Лучший результат: 0.90429


2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [16]:
df_rf_results = pd.DataFrame(grid_search_rf.cv_results_).sort_values(
    by=["rank_test_score", "param_max_depth"]
)

df_dt_table = df_dt_results[show_cols_dt]
df_dt_table.head(5)

,rank_test_score,mean_test_score,std_test_score,param_criterion,param_max_depth,param_class_weight
69,1,0.873865,0.025066,gini,21,balanced
73,2,0.873854,0.025018,gini,25,balanced
70,3,0.872378,0.025263,gini,22,balanced
71,4,0.872372,0.025179,gini,23,balanced
75,4,0.872372,0.025179,gini,27,balanced


## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.

In [17]:
n_estimators = (5, 10, 50, 100)
max_depth = range(1, 50)
criterion = ("entropy", "gini")
class_weight = ("balanced", None)
models = []

for est in tqdm(n_estimators, file=sys.stderr):
    for depth in max_depth:
        for cr in criterion:
            for w in class_weight:
                model = RandomForestClassifier(
                    random_state=21,
                    n_estimators=est,
                    max_depth=depth,
                    criterion=cr,
                    class_weight=w,
                    n_jobs=-1,
                )

                test_scores = cross_val_score(
                    model, X_train, y_train, cv=5, n_jobs=-1, scoring="accuracy"
                )

                mean_accuracy = test_scores.mean()
                std_accuracy = test_scores.std()

                models.append(
                    {
                        "n_estimators": est,
                        "max_depth": depth,
                        "criterion": cr,
                        "class_weight": w,
                        "mean_accuracy": mean_accuracy,
                        "std_accuracy": std_accuracy,
                        "model": model,
                    }
                )


  0%|          | 0/4 [00:00<?, ?it/s]

3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [18]:
RFmodels_df = pd.DataFrame(models).sort_values(by="mean_accuracy", ascending=False)
RFmodels_df.head(3).drop(columns=["model"]).style.hide_index()

n_estimators,max_depth,criterion,class_weight,mean_accuracy,std_accuracy
100,24,entropy,balanced,0.904293,0.012361
50,28,gini,None,0.904290,0.010961
100,29,entropy,balanced,0.904290,0.012156


In [19]:
pd.set_option("display.max_colwidth", None)
RFmodels_df["model"].head(1)

680    RandomForestClassifier(class_weight='balanced', criterion='entropy',\n                       max_depth=24, n_jobs=-1, random_state=21)
Name: model, dtype: object

## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.

In [20]:
best_model = RFmodels_df.loc[680, "model"]
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

2. Calculate the final accuracy.

In [21]:
final_accuracy = accuracy_score(y_pred, y_test)
print(f"Точность финальной модели Random Forest: \033[1;32m{final_accuracy:.5f}\033[0m")

Точность финальной модели Random Forest: 0.92604


In [22]:
notebook_total_time = time.time() - notebook_start_time
minutes = int(notebook_total_time // 60)
seconds = int(notebook_total_time % 60)

display(
    Markdown(
        f"""
---
**Время работы блокнота:** <span style="color:green">**{minutes} мин. {seconds} сек.**</span>

---
"""
    )
)


---
**Время работы блокнота:** <span style="color:green">**6 мин. 4 сек.**</span>

---
